# Portable CEB-IMDb block-skipping evaluation

This notebook partitions every DuckDB base table into consecutive row blocks, builds one infix fingerprint for each `(table, column, block)`, evaluates it without permitting false negatives, and writes the same core result files as `matrix-compression/block_skipping_results`.

The block fingerprint is the block-level form of the row substring fingerprint: values in one block are treated as one long, boundary-separated string; selected n-grams become fingerprint bits; and a block is a candidate when its mask contains every selected bit present in the query. The fingerprint is only a pre-filter—the original row predicate remains the source of truth. Query files use the existing `<table>.<column>_queries.txt` CSV format: predicate, optional matching-row count, optional matching percentage.

In [75]:
# Optional, once per environment:
# %pip install duckdb matplotlib

from __future__ import annotations

import csv
from datetime import datetime, timezone
import json
import math
import shutil
import unicodedata
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Iterable

import duckdb

# All relative paths are resolved from the directory containing the notebook.
PROJECT_ROOT = Path.cwd().resolve()
DATABASE_PATH = Path('/home/reese/block-fingerprints-project/block-skipping/data/workloads/ceb_imdb/ceb_imdb.duckdb')
QUERY_SOURCE_DIR = DATABASE_PATH.parent
RUN_DIR = PROJECT_ROOT / 'ceb_imdb_block_skipping_run'
QUERY_DIR = RUN_DIR / 'queries'
OUTPUT_DIR = RUN_DIR / 'block_skipping_results'
BLOCK_SIZE_ROWS = 16_384
WORKLOAD_NAME = 'ceb_imdb'

# Block-level infix fingerprint settings. Each requested width becomes one
# metadata version. Start with (16,); use e.g. (8, 16, 32, 64) for a trade-off sweep.
FINGERPRINT_WIDTHS = (16,)
FINGERPRINT_NGRAM_SIZE = 2
FINGERPRINT_ASCII_ONLY = True
# 'local_split_entropy' preserves the original recursive splitter.
# 'fingerprint_distribution_entropy' maximizes the entropy gain of the
# complete block-fingerprint distribution at every selection step.
FINGERPRINT_FEATURE_SELECTION = 'local_split_entropy'
FINGERPRINT_FEATURE_SELECTION_METHODS = {
    'local_split_entropy', 'fingerprint_distribution_entropy',
}
# Candidate-feature cutoff measured across blocks, not rows or occurrences.
# 1.0 considers every observed n-gram; 0.5 considers only n-grams present
# in at most 50% of blocks. See the detailed explanation below.
FINGERPRINT_MAX_BLOCK_FREQUENCY = 1.0
# Applied only by fingerprint_distribution_entropy. A value of 0.01 keeps
# candidates present in at least 1% of blocks; 0.0 keeps singleton grams.
FINGERPRINT_MIN_BLOCK_FREQUENCY = 0.0
FINGERPRINT_DIR = RUN_DIR / 'block_infix_fingerprints'
BLOCK_VALUE_SEPARATOR = chr(0)

# None means every target represented by a copied query file.
# Example: {('title', 'title'), ('name', 'name')}
# TARGETS: set[tuple[str, str]] | None = None
TARGETS = {("title", "title")}
QUERY_LIMIT: int | None = None # inlcude all queries

assert BLOCK_SIZE_ROWS > 0
assert FINGERPRINT_WIDTHS and all(width > 0 for width in FINGERPRINT_WIDTHS)
assert tuple(sorted(set(FINGERPRINT_WIDTHS))) == FINGERPRINT_WIDTHS
assert FINGERPRINT_NGRAM_SIZE > 0
assert FINGERPRINT_FEATURE_SELECTION in FINGERPRINT_FEATURE_SELECTION_METHODS
assert 0 <= FINGERPRINT_MIN_BLOCK_FREQUENCY <= FINGERPRINT_MAX_BLOCK_FREQUENCY <= 1
if not DATABASE_PATH.is_file():
    raise FileNotFoundError(f'CEB-IMDb DuckDB database not found: {DATABASE_PATH}')
print(f'Database: {DATABASE_PATH}')
print(f'Run output: {RUN_DIR}')

Database: /home/reese/block-fingerprints-project/block-skipping/data/workloads/ceb_imdb/ceb_imdb.duckdb
Run output: /home/reese/row-level-string-fingerprints/ceb_imdb_block_skipping_run


## Consecutively partition the database

Partition IDs are derived from the current physical `rowid` order: rows `0..block_size-1` form partition 0, and so on. Existing `partition_id` columns are replaced. Re-running the cell keeps the current row order and regenerates the same block boundaries.

In [76]:
def qi(identifier: str) -> str:
    """Quote and escape a DuckDB table or column identifier."""
    return chr(34) + identifier.replace(chr(34), chr(34) * 2) + chr(34)


def partition_database_consecutively(database_path: Path, block_size: int) -> None:
    if block_size <= 0:
        raise ValueError('block_size must be positive')
    with duckdb.connect(str(database_path)) as con:
        tables = [row[0] for row in con.execute(
            '''SELECT table_name FROM information_schema.tables
               WHERE table_schema = 'main' AND table_type = 'BASE TABLE'
                 AND table_name <> 'partition_metadata'
               ORDER BY table_name'''
        ).fetchall()]
        if not tables:
            raise ValueError('No base tables found in the database')
        for index, table_name in enumerate(tables, 1):
            table = qi(table_name)
            has_partition_id = bool(con.execute(
                '''SELECT count(*) FROM information_schema.columns
                   WHERE table_schema='main' AND table_name=? AND column_name='partition_id' ''',
                [table_name],
            ).fetchone()[0])
            source_columns = '* EXCLUDE (partition_id)' if has_partition_id else '*'
            con.execute(f'''
                CREATE OR REPLACE TABLE {table} AS
                WITH ordered AS (
                    SELECT {source_columns}, rowid AS __row_number
                    FROM {table}
                    ORDER BY rowid
                )
                SELECT * EXCLUDE (__row_number),
                       CAST(FLOOR(__row_number / {block_size}) AS INTEGER) AS partition_id
                FROM ordered
            ''')
            rows, blocks = con.execute(
                f'SELECT count(*), count(DISTINCT partition_id) FROM {table}'
            ).fetchone()
            print(f'[{index}/{len(tables)}] {table_name}: {rows:,} rows, {blocks:,} blocks')

# database already partitioned
# partition_database_consecutively(DATABASE_PATH, BLOCK_SIZE_ROWS)

## Copy the generated workload queries

This makes the run transferable and prevents the evaluation from modifying the source workload directory.

In [77]:
def query_target(path: Path) -> tuple[str, str]:
    suffix = '_queries.txt'
    if not path.name.endswith(suffix):
        raise ValueError(f'Unexpected query filename: {path.name}')
    table, separator, column = path.name[:-len(suffix)].rpartition('.')
    if not separator or not table or not column:
        raise ValueError(f'Expected <table>.<column>{suffix}: {path.name}')
    return table, column


def copy_queries(source_dir: Path, destination_dir: Path) -> list[Path]:
    sources = sorted(source_dir.glob('*_queries.txt'))
    if TARGETS is not None:
        sources = [path for path in sources if query_target(path) in TARGETS]
    if not sources:
        raise FileNotFoundError(f'No matching *_queries.txt files in {source_dir}')
    destination_dir.mkdir(parents=True, exist_ok=True)
    copied = []
    for source in sources:
        destination = destination_dir / source.name
        shutil.copy2(source, destination)
        copied.append(destination)
    return copied


QUERY_FILES = copy_queries(QUERY_SOURCE_DIR, QUERY_DIR)
print(f'Copied {len(QUERY_FILES)} query files to {QUERY_DIR}')

Copied 1 query files to /home/reese/row-level-string-fingerprints/ceb_imdb_block_skipping_run/queries


## Build the block-level infix fingerprint

This cell adapts the row-level substring/infix mask to blocks in five steps. It intentionally does not use the quantile prefix/suffix codes from `build_infix.py`: after concatenation those codes describe only the beginning and end of the entire block, so using them for CEB's arbitrary `%value%` predicates would cause false negatives.

1. **Concatenate a block logically.** For each target column, rows are streamed in `(partition_id, rowid)` order. The non-null values of one partition are joined using a NUL boundary marker, so only one block-sized string is held in Python at a time. The boundary prevents an n-gram from accidentally joining the end of one row to the start of the next.
2. **Extract block n-grams.** The notebook uses the other repository's `NGramGenerator` semantics exactly: normalize with Unicode NFKD, remove combining accent marks, lowercase, generate sliding character n-grams, and—by default—keep only ASCII grams. The only block-specific addition is removing grams containing the NUL row separator. Stored blocks and query predicates go through the same generator.
3. **Choose informative bits.** The configurable selector either uses the original local decision-tree entropy split or maximizes the entropy gain of the complete block-fingerprint distribution. Both use Python integer bitsets and favor n-grams whose presence separates blocks. Feature order is nested, so a 16-bit version is a prefix of a 32-bit version for the same method.
4. **Create one mask per block.** Bit `i` is one exactly when selected feature `i` occurs somewhere in the block's concatenated string. This is an OR-summary of all rows in the block. It can produce false positives because it forgets order and row identity, but it cannot remove a real match.
5. **Probe by containment.** A query receives the same selected-feature mask. A block survives when `(block_mask & query_mask) == query_mask`. If the query sets no selected bits—for example, it is shorter than the configured n-gram size—all blocks survive because the fingerprint has no exclusion evidence.

### What the block-frequency cutoffs mean

This value is a candidate-feature cutoff based on **block presence**, calculated independently for each target `(table, column)`. For an n-gram `g`, its block frequency is `number of blocks containing g / total number of blocks`. With 100 blocks, `1.0` admits n-grams found in at most 100 blocks, `0.5` admits those found in at most 50 blocks, and `0.1` admits those found in at most 10 blocks. It counts a block only once regardless of how many times the n-gram occurs inside that block.

At `1.0`, every observed n-gram allowed by the minimum cutoff enters the candidate pool and the entropy selector decides which features are useful. An n-gram present in every block has zero entropy, so it will not be selected even though it is eligible. Lower values reduce feature-selection work and force the selector toward rarer n-grams, but they can also leave more queries with a zero query mask and therefore less pruning. This setting cannot introduce false negatives: excluded or unselected n-grams simply provide no filtering evidence, and a zero mask conservatively keeps every block. Start with `1.0`; compare values such as `0.5` or `0.25` when measuring build time and pruning quality.

`FINGERPRINT_MIN_BLOCK_FREQUENCY` applies only to `fingerprint_distribution_entropy`. It removes candidates present in fewer than `ceil(minimum × B)` blocks. A value of `0.0` admits singleton n-grams; with 155 blocks, `0.01` requires at least 2 blocks and `0.05` requires at least 8. Raising the minimum can reduce work but can also make selection saturate earlier by removing the only n-grams that distinguish residual duplicate fingerprints. The local splitter ignores this minimum to preserve its original behavior.

The metadata is written as deterministic JSON, one file per configured width. The evaluator below independently computes exact matching partitions and raises immediately if this adapter ever produces a false negative.

## How entropy-based feature selection works (and where it stops scaling)

The selector works independently for each `(table, column)`, and its observations are **blocks**, not rows or query outcomes. Suppose there are `B` blocks. For every observed n-gram `g`, it first builds a `B`-bit presence vector `v_g`: bit `b` is one if `g` occurs anywhere in block `b`, otherwise zero. Repetitions within a block do not matter. The optional maximum-frequency cutoff removes `g` before selection when `popcount(v_g)` is above the configured fraction of `B`.

With `local_split_entropy`, selection constructs a small decision tree over block IDs. It starts with one unresolved group containing all blocks. For each still-available candidate n-gram and each current group `P`, it counts `h`, the blocks in `P` that contain the n-gram, and scores the possible split as:

`score(g, P) = |P| * H2(h / |P|)`, where `H2(p) = -p log2(p) - (1-p) log2(1-p)`.

Binary entropy is zero when the gram is in none or all of `P`, and maximal when it is in about half. The `|P|` multiplier gives a useful balanced split of a large group more weight than an equally balanced split of a tiny group. **Scores are separate for every `(candidate n-gram, current group P)` pair; they are not one global score per n-gram.** At each step the implementation chooses the single highest positive-scoring pair across all candidates and groups (lexical n-gram order breaks exact ties), records that pair's n-gram as the next fingerprint bit, removes the n-gram from the candidate set, and splits only that pair's group `P` into its absent and present subgroups. All other groups remain unchanged until a later iteration selects a split in one of them. It stops at the requested width or when no candidate can split any current group. Consequently, the first 16 selected grams are exactly the 16-bit version's features when a wider version is also built.

With `fingerprint_distribution_entropy`, the groups are instead the equivalence classes of complete fingerprints produced by all features selected so far. A candidate receives the sum of its split scores over every current group: `score(g) = sum_P |P| * H2(h_g,P / |P|)`. This is `B` times the increase in Shannon entropy obtained by appending the candidate bit. The best candidate is selected, and every group it separates is split. Selection stops when no candidate increases fingerprint-distribution entropy.

This is a **distributional** objective, not a direct search-quality objective. It does not know which predicates are common, how long they are, which n-grams appear in a given query workload, or whether two grams together identify the same row. It prefers grams that divide the existing blocks evenly. That is often a sensible way to spend a few bits, but it is only a proxy for reducing candidate blocks.

### Width: benefits and scaling limits

Adding bits can only add query constraints, so for the same selected-feature prefix it cannot increase the set of candidate blocks. In practice, the gain usually tapers off: selected grams can be correlated, popular predicates may contain none of the selected grams, and the OR mask cannot express that all evidence came from one row in the required order. More width also increases stored mask size approximately linearly (`B × width` bits before JSON/feature-name overhead), but greedy selection is substantially more expensive than encoding. At selection step `t`, `local_split_entropy` tests every candidate against at most `t + 1` tracked groups; `fingerprint_distribution_entropy` tests it against every current fingerprint class, at most `min(2^t, B)`. Large widths and candidate vocabularies can therefore make feature selection, rather than metadata storage, the bottleneck.

Wider fingerprints also do not repair the method's basic false-positive modes. A query's selected grams may occur in different rows of the same block, or in incompatible positions; the block still passes containment. Conversely, a query shorter than the n-gram size, or one whose grams were not selected, has a zero query mask and scans every block. The exact DuckDB `ILIKE` predicate is still required for correctness.

### N-gram size: 2-, 3-, and 5-grams

Increasing `n` normally makes an individual gram more specific, which can reduce accidental block matches. It also expands the observed vocabulary sharply (the theoretical ASCII space grows as `128^n`) and makes the presence vectors sparser. The observed corpus, rather than that theoretical maximum, determines the actual cost, but moving from 2 to 3 and especially 5 characters can greatly increase the candidate count `C`, the memory used by the `presence` dictionary, and the repeated candidate-by-group scans above. A permissive frequency cutoff such as `1.0` is most expensive in this setting; a lower cutoff caps candidates but may discard grams that real queries need.

- **2-grams:** very broad query coverage—any predicate of length two or more can contribute evidence—but common bigrams appear in many blocks and are weak filters. They keep the vocabulary relatively manageable, though common text columns can still have many candidates.

- **3-grams:** often a practical middle ground. They are more selective than bigrams, but every one- or two-character predicate becomes an all-block scan. Their larger vocabulary makes entropy selection slower and more sensitive to the frequency cutoff.

- **5-grams:** can be highly selective for long literals, but coverage becomes fragile: every predicate shorter than five characters has a zero mask, and a fixed-width feature list represents only a tiny fraction of the much larger 5-gram vocabulary. Many useful query grams will simply not be fingerprint bits. Sparse grams also tend to yield low-entropy, rare-block splits, which need not help the actual workload.

For a fair comparison, hold block size, normalization, workload, and candidate-frequency policy fixed; measure build time, peak memory, zero-mask-query rate, metadata size, and exact candidate-block reduction for each `(n, width)` pair. If large widths or 5-grams are required, practical mitigations are workload-aware feature scoring, prefiltering/capping candidates before the greedy loop, sampling blocks for selection, and a compact fixed-size hashed fingerprint. Each alternative changes the collision, reproducibility, or tuning trade-off, so it must retain the conservative rule that uncertain blocks are scanned.

In [78]:
@dataclass(frozen=True)
class FingerprintProbe:
    candidate_partition_ids: frozenset[int]
    width: int = 0
    ones: int = 0


@dataclass(frozen=True)
class FingerprintVersion:
    metadata_version: int
    merge_step: str
    metadata_file: str
    metadata_size_bytes: int
    mean_matrix_rows: float
    ngram_size: int | None
    probe: Callable[[str, str, str], FingerprintProbe]


class NGramGenerator:
    """Generate canonical character n-grams using the source repository's rules."""

    def __init__(self, n: int, ascii_only: bool = True):
        if n <= 0:
            raise ValueError('n must be positive for n-gram generation.')
        self.n = n
        self.ascii_only = ascii_only

    def generate(self, text: str):
        # This matches src/NGramGenerator.py: NFKD decomposition first, then
        # removal of combining marks, followed by lowercase normalization.
        text = self.strip_accents(text).lower()
        if len(text) < self.n:
            return []
        ngrams = [text[i:i + self.n] for i in range(len(text) - self.n + 1)]
        if self.ascii_only:
            return [ngram for ngram in ngrams if ngram.isascii()]
        return ngrams

    @staticmethod
    def strip_accents(text: str) -> str:
        return ''.join(
            character
            for character in unicodedata.normalize('NFKD', text)
            if not unicodedata.combining(character)
        )


NGRAM_GENERATOR = NGramGenerator(
    FINGERPRINT_NGRAM_SIZE, ascii_only=FINGERPRINT_ASCII_ONLY
)


def iter_fingerprint_ngrams(text: str):
    # NUL marks row boundaries. The source generator is unchanged; this final
    # filter is specific to concatenated blocks and prevents cross-row grams.
    for gram in NGRAM_GENERATOR.generate(text):
        if BLOCK_VALUE_SEPARATOR not in gram:
            yield gram


def concatenate_partition_ngrams(
    con: duckdb.DuckDBPyConnection,
    table: str,
    column: str,
) -> dict[int, frozenset[str]]:
    # Values stay unnormalized until NGramGenerator handles them. Streaming rows
    # keeps memory bounded to one concatenated partition plus its n-gram set.
    cursor = con.execute(
        f'''SELECT partition_id, CAST({qi(column)} AS VARCHAR)
            FROM {qi(table)}
            ORDER BY partition_id, rowid'''
    )
    partition_grams: dict[int, frozenset[str]] = {}
    active_partition: int | None = None
    values: list[str] = []

    def finish_partition() -> None:
        if active_partition is None:
            return
        concatenated = BLOCK_VALUE_SEPARATOR.join(values)
        partition_grams[active_partition] = frozenset(
            iter_fingerprint_ngrams(concatenated)
        )

    while True:
        batch = cursor.fetchmany(50_000)
        if not batch:
            break
        for raw_partition, value in batch:
            partition_id = int(raw_partition)
            if active_partition is None:
                active_partition = partition_id
            elif partition_id != active_partition:
                finish_partition()
                active_partition = partition_id
                values = []
            if value is not None:
                values.append(str(value))
    finish_partition()
    return partition_grams


def binary_entropy(successes: int, population: int) -> float:
    if successes <= 0 or successes >= population:
        return 0.0
    probability = successes / population
    return -(probability * math.log2(probability)
             + (1 - probability) * math.log2(1 - probability))


def block_ngram_candidates(
    partition_grams: dict[int, frozenset[str]],
) -> tuple[int, dict[str, int]]:
    # Map every gram to a Python-int presence bitmap over the target's blocks.
    partition_ids = sorted(partition_grams)
    position = {partition_id: index for index, partition_id in enumerate(partition_ids)}
    presence: dict[str, int] = {}
    for partition_id, grams in partition_grams.items():
        bit = 1 << position[partition_id]
        for gram in grams:
            presence[gram] = presence.get(gram, 0) | bit

    block_count = len(partition_ids)
    if block_count == 0:
        return 0, {}
    max_frequency_count = max(1, math.floor(
        FINGERPRINT_MAX_BLOCK_FREQUENCY * block_count
    ))
    min_frequency_count = 1
    if FINGERPRINT_FEATURE_SELECTION == 'fingerprint_distribution_entropy':
        min_frequency_count = max(1, math.ceil(
            FINGERPRINT_MIN_BLOCK_FREQUENCY * block_count
        ))
    candidates = {
        gram: mask for gram, mask in sorted(presence.items())
        if min_frequency_count <= mask.bit_count() <= max_frequency_count
    }
    return block_count, candidates


def select_local_split_entropy_features(
    block_count: int, candidates: dict[str, int], limit: int,
) -> list[str]:
    # Preserve the original recursive entropy selector: choose the best
    # (feature, one current group) split, then split only that group.
    selected: list[str] = []
    groups = [(1 << block_count) - 1]
    while candidates and len(selected) < limit:
        best_feature = None
        best_group_index = -1
        best_score = 0.0
        for gram, gram_mask in candidates.items():
            for group_index, group_mask in enumerate(groups):
                size = group_mask.bit_count()
                hits = (gram_mask & group_mask).bit_count()
                score = binary_entropy(hits, size) * size
                if (score > best_score or
                        (score == best_score and score > 0 and
                         (best_feature is None or gram < best_feature))):
                    best_feature = gram
                    best_group_index = group_index
                    best_score = score
        if best_feature is None:
            break
        selected.append(best_feature)
        feature_mask = candidates.pop(best_feature)
        parent = groups.pop(best_group_index)
        absent, present = parent & ~feature_mask, parent & feature_mask
        groups.extend(group for group in (absent, present) if group)
    return selected


def select_fingerprint_distribution_entropy_features(
    block_count: int, candidates: dict[str, int], limit: int,
) -> list[str]:
    # Each group contains blocks with the same fingerprint under the already
    # selected features. The score is block_count times the entropy gain of
    # appending a candidate bit to the complete fingerprint distribution.
    selected: list[str] = []
    groups = [(1 << block_count) - 1]
    while candidates and len(selected) < limit:
        best_feature = None
        best_score = 0.0
        for gram, gram_mask in candidates.items():
            score = math.fsum(
                group_mask.bit_count() * binary_entropy(
                    (gram_mask & group_mask).bit_count(),
                    group_mask.bit_count(),
                )
                for group_mask in groups
            )
            if (score > best_score or
                    (score == best_score and score > 0 and
                     (best_feature is None or gram < best_feature))):
                best_feature = gram
                best_score = score
        if best_feature is None:
            break
        selected.append(best_feature)
        feature_mask = candidates.pop(best_feature)
        next_groups = []
        for group_mask in groups:
            absent, present = group_mask & ~feature_mask, group_mask & feature_mask
            next_groups.extend(group for group in (absent, present) if group)
        groups = next_groups
    return selected


def select_block_features(
    partition_grams: dict[int, frozenset[str]],
    limit: int,
) -> list[str]:
    block_count, candidates = block_ngram_candidates(partition_grams)
    if block_count == 0:
        return []
    if FINGERPRINT_FEATURE_SELECTION == 'local_split_entropy':
        return select_local_split_entropy_features(block_count, candidates, limit)
    if FINGERPRINT_FEATURE_SELECTION == 'fingerprint_distribution_entropy':
        return select_fingerprint_distribution_entropy_features(
            block_count, candidates, limit
        )
    raise ValueError(
        f'Unknown fingerprint feature-selection method: '
        f'{FINGERPRINT_FEATURE_SELECTION!r}'
    )


def encode_block(grams: frozenset[str], features: list[str]) -> int:
    return sum(1 << index for index, feature in enumerate(features)
               if feature in grams)


def encode_query(predicate: str, features: tuple[str, ...]) -> int:
    # Query and stored text use exactly the same canonical n-gram generator.
    # If no selected query grams remain, the zero mask scans every block.
    query_grams = frozenset(NGRAM_GENERATOR.generate(predicate))
    return sum(1 << index for index, feature in enumerate(features)
               if feature in query_grams)


def fingerprint_payload_size_bytes(
    width: int, profiles: dict[tuple[str, str], dict],
) -> int:
    # Report only fixed-width block masks. Feature names, partition IDs,
    # JSON syntax, and all other serialization overhead are excluded.
    bytes_per_fingerprint = math.ceil(width / 8)
    fingerprint_count = sum(
        len(profile['block_masks']) for profile in profiles.values()
    )
    return bytes_per_fingerprint * fingerprint_count


def write_fingerprint_metadata(
    path: Path,
    width: int,
    profiles: dict[tuple[str, str], dict],
) -> None:
    hex_digits = max(1, math.ceil(width / 4))
    payload = {
        'schema_version': 1,
        'representation': 'one_concatenated_infix_mask_per_block',
        'block_size_rows': BLOCK_SIZE_ROWS,
        'fingerprint_width': width,
        'fingerprint_bytes_per_block': math.ceil(width / 8),
        'fingerprint_payload_size_bytes': fingerprint_payload_size_bytes(
            width, profiles
        ),
        'ngram_size': FINGERPRINT_NGRAM_SIZE,
        'feature_selection_method': FINGERPRINT_FEATURE_SELECTION,
        'min_block_frequency': FINGERPRINT_MIN_BLOCK_FREQUENCY,
        'max_block_frequency': FINGERPRINT_MAX_BLOCK_FREQUENCY,
        'ascii_only': FINGERPRINT_ASCII_ONLY,
        'normalization': 'NFKD-strip-accents-lower',
        'targets': {
            f'{table}.{column}': {
                'features': list(profile['features']),
                'feature_diagnostics': [
                    {
                        'bit_index': bit_index,
                        'ngram': feature,
                        'block_presence_count': profile['feature_block_counts'][bit_index],
                        'total_block_count': profile['block_count'],
                        'block_frequency': (
                            profile['feature_block_counts'][bit_index]
                            / profile['block_count']
                            if profile['block_count'] else 0.0
                        ),
                    }
                    for bit_index, feature in enumerate(profile['features'])
                ],
                'blocks': [
                    {'partition_id': partition_id,
                     'mask_hex': format(mask, f'0{hex_digits}x')}
                    for partition_id, mask in sorted(profile['block_masks'].items())
                ],
            }
            for (table, column), profile in sorted(profiles.items())
        },
    }
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as handle:
        json.dump(payload, handle, ensure_ascii=True, sort_keys=True, separators=(',', ':'))
        handle.write('\n')


def load_fingerprint_versions(
    con: duckdb.DuckDBPyConnection,
    targets: set[tuple[str, str]],
) -> list[FingerprintVersion]:
    max_width = max(FINGERPRINT_WIDTHS)
    full_profiles: dict[tuple[str, str], dict] = {}
    for target_index, (table, column) in enumerate(sorted(targets), 1):
        print(f'Building concatenated block strings [{target_index}/{len(targets)}]: {table}.{column}')
        partition_grams = concatenate_partition_ngrams(con, table, column)
        features = select_block_features(partition_grams, max_width)
        full_profiles[(table, column)] = {
            'features': tuple(features),
            'feature_block_counts': tuple(
                sum(feature in grams for grams in partition_grams.values())
                for feature in features
            ),
            'block_count': len(partition_grams),
            'block_masks': {partition_id: encode_block(grams, features)
                            for partition_id, grams in partition_grams.items()},
        }
        print(
            f'  {len(partition_grams):,} blocks; selected '
            f'{len(features):,}/{max_width} features with '
            f'{FINGERPRINT_FEATURE_SELECTION}'
        )
        print(f'  feature order: {features}')

    selected_feature_count = max(
        (len(profile['features']) for profile in full_profiles.values()),
        default=0,
    )
    selection_saturated = all(
        len(profile['features']) < max_width for profile in full_profiles.values()
    )
    version_widths = FINGERPRINT_WIDTHS
    if selection_saturated:
        last_version_index = next(
            index for index, width in enumerate(FINGERPRINT_WIDTHS)
            if width >= selected_feature_count
        )
        version_widths = FINGERPRINT_WIDTHS[:last_version_index + 1]
        print(
            f'  selection saturated at {selected_feature_count} features; '
            f'stopping width sweep at {version_widths[-1]} bits'
        )

    versions: list[FingerprintVersion] = []
    diagnostic_rows = []
    for version_id, width in enumerate(version_widths):
        width_mask = (1 << width) - 1
        profiles = {
            target: {
                'features': profile['features'][:width],
                'feature_block_counts': profile['feature_block_counts'][:width],
                'block_count': profile['block_count'],
                'block_masks': {partition_id: mask & width_mask
                                for partition_id, mask in profile['block_masks'].items()},
            }
            for target, profile in full_profiles.items()
        }
        metadata_path = FINGERPRINT_DIR / f'block_infix_fingerprint_v{version_id:03d}.json'
        write_fingerprint_metadata(metadata_path, width, profiles)
        for (table, column), profile in sorted(profiles.items()):
            for bit_index, feature in enumerate(profile['features']):
                presence_count = profile['feature_block_counts'][bit_index]
                diagnostic_rows.append({
                    'metadata_version': version_id,
                    'metadata_file': metadata_path.name,
                    'fingerprint_width': width,
                    'feature_selection_method': FINGERPRINT_FEATURE_SELECTION,
                    'ngram_size': FINGERPRINT_NGRAM_SIZE,
                    'min_block_frequency': FINGERPRINT_MIN_BLOCK_FREQUENCY,
                    'max_block_frequency': FINGERPRINT_MAX_BLOCK_FREQUENCY,
                    'table_name': table, 'column_name': column,
                    'bit_index': bit_index, 'ngram': feature,
                    'block_presence_count': presence_count,
                    'total_block_count': profile['block_count'],
                    'block_frequency': (
                        presence_count / profile['block_count']
                        if profile['block_count'] else 0.0
                    ),
                })

        def make_probe(version_profiles, fingerprint_width):
            def probe(table: str, column: str, predicate: str) -> FingerprintProbe:
                profile = version_profiles[(table, column)]
                query_mask = encode_query(str(predicate or ''), profile['features'])
                candidates = frozenset(
                    partition_id
                    for partition_id, block_mask in profile['block_masks'].items()
                    if (block_mask & query_mask) == query_mask
                )
                return FingerprintProbe(
                    candidate_partition_ids=candidates,
                    width=fingerprint_width,
                    ones=query_mask.bit_count(),
                )
            return probe

        versions.append(FingerprintVersion(
            metadata_version=version_id,
            merge_step=(
                f'concatenated_block_infix_{FINGERPRINT_FEATURE_SELECTION}_{width}bit'
            ),
            metadata_file=metadata_path.name,
            metadata_size_bytes=fingerprint_payload_size_bytes(width, profiles),
            mean_matrix_rows=1.0,
            ngram_size=FINGERPRINT_NGRAM_SIZE,
            probe=make_probe(profiles, width),
        ))

    diagnostic_columns = [
        'metadata_version', 'metadata_file', 'fingerprint_width',
        'feature_selection_method', 'ngram_size', 'min_block_frequency',
        'max_block_frequency',
        'table_name', 'column_name', 'bit_index', 'ngram',
        'block_presence_count', 'total_block_count', 'block_frequency',
    ]
    diagnostics_path = OUTPUT_DIR / 'selected_ngram_diagnostics.csv'
    diagnostics_path.parent.mkdir(parents=True, exist_ok=True)
    with diagnostics_path.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=diagnostic_columns)
        writer.writeheader()
        writer.writerows(diagnostic_rows)
    print(f'Wrote selected n-gram diagnostics: {diagnostics_path}')
    return versions

## Evaluation and version-aware exports

The evaluator loads the copied workloads, builds each configured fingerprint width using the selected workload-agnostic entropy method over all eligible block n-grams, computes exact matching blocks in DuckDB, and rejects any false negative before writing results. Reported metadata size is only the packed fixed-width fingerprint payload: `number of block fingerprints × ceil(width / 8)` bytes. It excludes feature strings, partition IDs, JSON syntax, and other serialization overhead. Each run also writes `selected_ngram_diagnostics.csv`; every row identifies a selected bit and reports how many target blocks contain that n-gram. The same diagnostics are embedded in each metadata JSON file. `selected_ngram_block_frequency_histograms.pdf` shows one histogram per emitted width using the fraction of blocks containing each selected n-gram. Outputs and plots compare configured fingerprint versions; they do not assume a nonexistent before/after fuzzy-merging stage.


In [79]:
RESULT_COLUMNS = [
    'query_id', 'workload_file', 'workload_row_number', 'table_name',
    'column_name', 'predicate_value', 'expected_matching_rows',
    'expected_matching_percent', 'query_ngram_count',
    'query_fingerprint_width', 'query_fingerprint_ones',
    'metadata_version', 'merge_step', 'metadata_file',
    'partition_file_count', 'metadata_total_size_bytes',
    'ground_truth_partition_count', 'metadata_partition_count',
    'pruned_partition_count', 'false_positive_partition_count',
    'false_negative_partition_count',
]
SKIPPED_COLUMNS = ['workload_file', 'table_name', 'column_name', 'reason', 'query_count']


def write_csv(path: Path, columns: list[str], rows: Iterable[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=columns, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)


def normalized_ngrams(text: str) -> frozenset[str]:
    return frozenset(NGRAM_GENERATOR.generate(str(text)))


def escaped_like(value: str) -> str:
    escape = chr(92)
    return '%' + value.replace(escape, escape * 2).replace('%', escape + '%').replace('_', escape + '_') + '%'


def target_exists(con, table: str, column: str) -> bool:
    return bool(con.execute(
        '''SELECT count(*) FROM information_schema.columns
           WHERE table_schema='main' AND table_name=? AND column_name=?''',
        [table, column],
    ).fetchone()[0])


def load_workload(con) -> tuple[list[dict], list[dict]]:
    queries, skipped = [], []
    for path in QUERY_FILES:
        table, column = query_target(path)
        with path.open(newline='', encoding='utf-8') as handle:
            source_rows = [(number, row) for number, row in enumerate(csv.reader(handle), 1) if row]
        if not target_exists(con, table, column):
            skipped.append({
                'workload_file': path.name, 'table_name': table, 'column_name': column,
                'reason': 'target_not_in_database', 'query_count': len(source_rows),
            })
            continue
        for row_number, row in source_rows:
            if QUERY_LIMIT is not None and len(queries) >= QUERY_LIMIT:
                return queries, skipped
            queries.append({
                'query_id': len(queries), 'workload_file': path.name,
                'workload_row_number': row_number, 'table_name': table,
                'column_name': column, 'predicate_value': row[0],
                'expected_matching_rows': row[1] if len(row) > 1 else '',
                'expected_matching_percent': row[2] if len(row) > 2 else '',
            })
    if not queries:
        raise ValueError('No evaluable workload queries')
    return queries, skipped


def ground_truth_partitions(con, query: dict) -> frozenset[int]:
    rows = con.execute(
        f'''SELECT DISTINCT partition_id FROM {qi(query['table_name'])}
            WHERE CAST({qi(query['column_name'])} AS VARCHAR) ILIKE ? ESCAPE '\\'
            ORDER BY partition_id''',
        [escaped_like(query['predicate_value'])],
    ).fetchall()
    return frozenset(int(row[0]) for row in rows)


def assert_no_false_negatives(label, queries, ground_truths, matches) -> None:
    failures = []
    for query in queries:
        query_id = query['query_id']
        missing = ground_truths[query_id] - matches[query_id]
        if missing:
            failures.append((query_id, query['predicate_value'], sorted(missing)))
    if failures:
        sample = '; '.join(
            f'query_id={query_id} predicate={predicate!r} missed={missing[:20]}'
            for query_id, predicate, missing in failures[:10]
        )
        raise RuntimeError(
            f'FATAL ERROR: FINGERPRINT PRUNING IS INCORRECT for {label}. '
            f'{len(failures)} queries have false negatives. {sample}'
        )


def evaluate_fingerprints():
    with duckdb.connect(str(DATABASE_PATH), read_only=True) as con:
        queries, skipped = load_workload(con)
        targets = {(q['table_name'], q['column_name']) for q in queries}
        partition_ids = {
            target: frozenset(int(row[0]) for row in con.execute(
                f'SELECT DISTINCT partition_id FROM {qi(target[0])} ORDER BY partition_id'
            ).fetchall())
            for target in targets
        }
        ground_truths = {
            q['query_id']: ground_truth_partitions(con, q) for q in queries
        }
        versions = load_fingerprint_versions(con, targets)
        if not versions:
            raise ValueError('The fingerprint builder returned no versions')

        matches_by_version = {}
        result_rows = []
        for version in sorted(versions, key=lambda item: item.metadata_version):
            if version.metadata_size_bytes < 0:
                raise ValueError('Fingerprint metadata size cannot be negative')
            matches = {}
            probes = {}
            for query in queries:
                query_id = query['query_id']
                target = (query['table_name'], query['column_name'])
                probe = version.probe(*target, query['predicate_value'])
                candidates = frozenset(int(value) for value in probe.candidate_partition_ids)
                if not candidates <= partition_ids[target]:
                    raise ValueError(
                        f'Fingerprint version {version.metadata_version} returned invalid '
                        f'partition IDs for {target}: {sorted(candidates - partition_ids[target])}'
                    )
                matches[query_id] = candidates
                probes[query_id] = probe
            assert_no_false_negatives(
                f'version {version.metadata_version}', queries, ground_truths, matches
            )
            matches_by_version[version.metadata_version] = matches

            for query in queries:
                query_id = query['query_id']
                target = (query['table_name'], query['column_name'])
                truth = ground_truths[query_id]
                candidates = matches[query_id]
                probe = probes[query_id]
                result_rows.append({
                    **query,
                    'query_ngram_count': len(normalized_ngrams(query['predicate_value'])),
                    'query_fingerprint_width': probe.width,
                    'query_fingerprint_ones': probe.ones,
                    'metadata_version': version.metadata_version,
                    'merge_step': version.merge_step,
                    'metadata_file': version.metadata_file,
                    'partition_file_count': len(partition_ids[target]),
                    'metadata_total_size_bytes': version.metadata_size_bytes,
                    'ground_truth_partition_count': len(truth),
                    'metadata_partition_count': len(candidates),
                    'pruned_partition_count': len(partition_ids[target]) - len(candidates),
                    'false_positive_partition_count': len(candidates - truth),
                    'false_negative_partition_count': len(truth - candidates),
                })
            print(f'Evaluated version {version.metadata_version}: {len(queries)} queries')

    results_path = OUTPUT_DIR / 'results.csv'
    write_csv(results_path, RESULT_COLUMNS, result_rows)
    write_csv(OUTPUT_DIR / 'skipped_workloads.csv', SKIPPED_COLUMNS, skipped)
    print(f'Wrote evaluation metrics: {results_path}')
    return queries, skipped, ground_truths, versions, partition_ids, matches_by_version, result_rows


#(QUERIES, SKIPPED, GROUND_TRUTHS, FINGERPRINT_VERSIONS, PARTITION_IDS,
# MATCHES_BY_VERSION, RESULT_ROWS) = evaluate_fingerprints()


In [80]:
METRICS_COLUMNS = [
    'result_file', 'workload', 'data_structure', 'block_size_rows',
    'ngram_size', 'query_mode', 'configured_query_count', 'partition_count',
    'average_metadata_size_in_bytes', 'mean_unneccesary_block_read_ratio',
    'mean_false_block_reads_factor', 'mean_absolute_values',
    'included_query_count', 'skipped_query_count', 'total_query_count',
    'total_num_skipped_blocks',
]


def mean(values: list[float]):
    return '' if not values else math.fsum(values) / len(values)


def create_exports(results, versions, partition_ids) -> None:
    versions_by_id = {version.metadata_version: version for version in versions}
    metrics_rows, size_rows, ratio_rows, manifest_points = [], [], [], []
    for version_id in sorted(versions_by_id):
        version = versions_by_id[version_id]
        rows = [row for row in results if row['metadata_version'] == version_id]
        ratios, factors, absolutes = [], [], []
        skipped_queries = 0
        for row in rows:
            total = len(partition_ids[(row['table_name'], row['column_name'])])
            truth = row['ground_truth_partition_count']
            false_positives = row['false_positive_partition_count']
            skippable = total - truth
            if skippable == 0:
                skipped_queries += 1
                continue
            ratios.append(false_positives / skippable)
            if truth > 0:
                factors.append(false_positives / truth)
            absolutes.append(false_positives * BLOCK_SIZE_ROWS)

        represented_targets = {(row['table_name'], row['column_name']) for row in rows}
        matrix_count = sum(len(partition_ids[target]) for target in represented_targets)
        total_scanned = sum(row['metadata_partition_count'] for row in rows)
        total_truth = sum(row['ground_truth_partition_count'] for row in rows)
        total_possible = sum(row['partition_file_count'] for row in rows)
        total_skipped = total_possible - total_scanned
        metrics = {
            'metadata_size_bytes': version.metadata_size_bytes,
            'metadata_size_mib': version.metadata_size_bytes / 2**20,
            'partition_count': matrix_count,
            'query_count': len(rows),
            'total_skipped_partitions': total_skipped,
            'false_positive_partition_count': sum(
                row['false_positive_partition_count'] for row in rows
            ),
            'false_negative_partition_count': sum(
                row['false_negative_partition_count'] for row in rows
            ),
            'zero_bit_query_count': sum(
                row['query_fingerprint_ones'] == 0 for row in rows
            ),
        }
        metrics_rows.append({
            'result_file': version.metadata_file, 'workload': WORKLOAD_NAME,
            'data_structure': 'fingerprintMatrix', 'block_size_rows': BLOCK_SIZE_ROWS,
            'ngram_size': '' if version.ngram_size is None else version.ngram_size,
            'query_mode': 'query_file', 'configured_query_count': QUERY_LIMIT or '',
            'partition_count': matrix_count,
            'average_metadata_size_in_bytes': (
                version.metadata_size_bytes / matrix_count if matrix_count else ''
            ),
            'mean_unneccesary_block_read_ratio': mean(ratios),
            'mean_false_block_reads_factor': mean(factors),
            'mean_absolute_values': mean(absolutes),
            'included_query_count': len(ratios), 'skipped_query_count': skipped_queries,
            'total_query_count': len(rows), 'total_num_skipped_blocks': total_skipped,
        })
        size_rows.append({
            'metadata_version': version_id, 'merge_step': version.merge_step,
            'metadata_file': version.metadata_file,
            'metadata_file_size_bytes': version.metadata_size_bytes,
            'metadata_file_size_mib': f'{version.metadata_size_bytes / 2**20:.6f}',
            'total_skipped_partitions': total_skipped,
            'total_scanned_partitions': total_scanned,
            'total_ground_truth_partitions': total_truth,
            'query_count': len(rows),
        })
        ratio_rows.append({
            'metadata_version': version_id, 'merge_step': version.merge_step,
            'metadata_file': version.metadata_file,
            'metadata_file_size_bytes': version.metadata_size_bytes,
            'metadata_file_size_kb': f'{version.metadata_size_bytes / 1024:.8f}',
            'mean_metadata_file_size_kb_per_partition': (
                f'{version.metadata_size_bytes / 1024 / matrix_count:.8f}' if matrix_count else ''
            ),
            'mean_matrix_rows': f'{version.mean_matrix_rows:.8f}',
            'partition_count': matrix_count,
            'mean_unnecessary_block_read_ratio': (
                '' if not ratios else f'{mean(ratios):.8f}'
            ),
            'ratio_query_count': len(ratios), 'query_count': len(rows),
            'zero_bit_query_count': metrics['zero_bit_query_count'],
        })
        manifest_points.append({
            'id': f'block-infix-v{version_id:03d}',
            'title': version.merge_step,
            'technique': 'concatenated_block_infix_fingerprint',
            'parent_id': None,
            'parameters': {
                'fingerprint_width': next(
                    (row['query_fingerprint_width'] for row in rows), 0
                ),
                'feature_selection_method': FINGERPRINT_FEATURE_SELECTION,
                'ngram_size': version.ngram_size,
                'block_size_rows': BLOCK_SIZE_ROWS,
                'min_block_frequency': FINGERPRINT_MIN_BLOCK_FREQUENCY,
                'max_block_frequency': FINGERPRINT_MAX_BLOCK_FREQUENCY,
                'feature_candidates': 'frequency_filtered_observed_block_ngrams',
            },
            'metadata_dir': str(FINGERPRINT_DIR.resolve()),
            'results_path': str((OUTPUT_DIR / 'results.csv').resolve()),
            'metrics': metrics,
        })

    baseline = next((float(row['mean_unnecessary_block_read_ratio']) for row in ratio_rows
                     if row['metadata_version'] == 0 and row['mean_unnecessary_block_read_ratio'] != ''), None)
    for row in ratio_rows:
        value = row['mean_unnecessary_block_read_ratio']
        row['unnecessary_block_read_ratio_increase_from_v000'] = (
            '' if value == '' or baseline is None else f'{float(value) - baseline:.8f}'
        )

    write_csv(OUTPUT_DIR / 'metrics_summary.csv', METRICS_COLUMNS, metrics_rows)
    write_csv(OUTPUT_DIR / 'size_pruning_tradeoff_summary.csv', [
        'metadata_version', 'merge_step', 'metadata_file', 'metadata_file_size_bytes',
        'metadata_file_size_mib', 'total_skipped_partitions', 'total_scanned_partitions',
        'total_ground_truth_partitions', 'query_count'], size_rows)
    write_csv(OUTPUT_DIR / 'mean_rows_unnecessary_block_read_ratio_summary.csv', [
        'metadata_version', 'merge_step', 'metadata_file', 'metadata_file_size_bytes',
        'metadata_file_size_kb', 'mean_metadata_file_size_kb_per_partition',
        'mean_matrix_rows', 'partition_count', 'mean_unnecessary_block_read_ratio',
        'unnecessary_block_read_ratio_increase_from_v000', 'ratio_query_count',
        'query_count', 'zero_bit_query_count'], ratio_rows)
    (RUN_DIR / 'experiment_manifest.json').write_text(
        json.dumps({'format_version': 1, 'points': manifest_points}, indent=2, sort_keys=True) + '\n',
        encoding='utf-8',
    )

    import matplotlib.pyplot as plt
    from matplotlib.ticker import PercentFormatter

    sizes_mib = [float(row['metadata_file_size_mib']) for row in size_rows]
    skipped = [row['total_skipped_partitions'] for row in size_rows]
    labels = [f"v{row['metadata_version']}" for row in size_rows]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(sizes_mib, skipped, s=90, color='#2563eb', edgecolor='#172554')
    for label, x_value, y_value in zip(labels, sizes_mib, skipped):
        ax.annotate(label, (x_value, y_value), textcoords='offset points', xytext=(7, 7))
    ax.set_xlabel('Metadata Size (MiB)')
    ax.set_ylabel('Total Skipped Partitions Across Workload')
    ax.set_title('Metadata Size vs. Partition Pruning')
    ax.set_xlim(left=0); ax.set_ylim(bottom=0); ax.grid(alpha=.3); fig.tight_layout()
    fig.savefig(OUTPUT_DIR / 'size_pruning_tradeoff.pdf'); plt.close(fig)

    plottable = [row for row in ratio_rows if row['mean_unnecessary_block_read_ratio'] != '']
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(
        [float(row['mean_metadata_file_size_kb_per_partition']) for row in plottable],
        [float(row['mean_unnecessary_block_read_ratio']) for row in plottable],
        s=90, color='#2563eb', edgecolor='#172554',
    )
    for row in plottable:
        ax.annotate(
            f"v{row['metadata_version']}",
            (float(row['mean_metadata_file_size_kb_per_partition']),
             float(row['mean_unnecessary_block_read_ratio'])),
            textcoords='offset points', xytext=(7, 7),
        )
    ax.set_xlabel('Mean Metadata Size per Partition Matrix (kB)')
    ax.set_ylabel('Mean Unnecessary Block Read Ratio')
    ax.set_title('Metadata Size vs. Mean Unnecessary Block Read Ratio')
    ax.set_xlim(left=0); ax.set_ylim(bottom=0); ax.grid(alpha=.3); fig.tight_layout()
    fig.savefig(OUTPUT_DIR / 'mean_rows_unnecessary_block_read_ratio.pdf'); plt.close(fig)

    diagnostic_rows = []
    diagnostics_path = OUTPUT_DIR / 'selected_ngram_diagnostics.csv'
    if diagnostics_path.is_file():
        with diagnostics_path.open(newline='', encoding='utf-8') as handle:
            diagnostic_rows = list(csv.DictReader(handle))
    plot_versions = sorted(versions, key=lambda version: version.metadata_version)
    panel_columns = min(3, max(1, len(plot_versions)))
    panel_rows = math.ceil(len(plot_versions) / panel_columns)
    fig, axes = plt.subplots(
        panel_rows, panel_columns, squeeze=False,
        figsize=(4.5 * panel_columns, 3.4 * panel_rows),
        sharex=True, sharey=True,
    )
    histogram_bins = [index / 20 for index in range(21)]
    for ax, version in zip(axes.flat, plot_versions):
        version_diagnostics = [
            row for row in diagnostic_rows
            if int(row['metadata_version']) == version.metadata_version
        ]
        feature_frequencies = [
            float(row['block_frequency'])
            for row in version_diagnostics
        ]
        version_results = [
            row for row in results
            if row['metadata_version'] == version.metadata_version
        ]
        width = next(
            (row['query_fingerprint_width'] for row in version_results), 0
        )
        if feature_frequencies:
            ax.hist(
                feature_frequencies, bins=histogram_bins,
                weights=[1 / len(feature_frequencies)] * len(feature_frequencies),
                color='#2563eb', edgecolor='#172554', alpha=.85,
            )
        else:
            ax.text(.5, .5, 'No selected features', ha='center', va='center',
                    transform=ax.transAxes)
        ax.set_title(f'{width}-bit ({len(feature_frequencies)} features)')
        ax.set_xlim(0, 1); ax.set_xticks([index / 10 for index in range(11)])
        ax.set_ylim(0, 1); ax.grid(axis='y', alpha=.25)
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
    for ax in list(axes.flat)[len(plot_versions):]:
        ax.set_visible(False)
    fig.supxlabel('Block presence frequency')
    fig.supylabel('Share of selected n-grams')
    fig.suptitle(
        f'Selected n-gram block-frequency distributions\n'
        f'{FINGERPRINT_FEATURE_SELECTION}; n={FINGERPRINT_NGRAM_SIZE}; '
        f'frequency=[{FINGERPRINT_MIN_BLOCK_FREQUENCY:g}, '
        f'{FINGERPRINT_MAX_BLOCK_FREQUENCY:g}]'
    )
    fig.tight_layout(rect=(0, 0, 1, .94))
    fig.savefig(OUTPUT_DIR / 'selected_ngram_block_frequency_histograms.pdf')
    plt.close(fig)
    print(f'Wrote version-aware evaluation outputs to {OUTPUT_DIR}')


# create_exports(RESULT_ROWS, FINGERPRINT_VERSIONS, PARTITION_IDS)
# sorted(path.name for path in OUTPUT_DIR.iterdir())


In [81]:
# Sweep feature-selection method, fingerprint width, n-gram size, and the
# minimum/maximum block-frequency cutoffs. Widths are nested versions, so
# each (method, n-gram, min frequency, max frequency) corpus is scanned once.
SWEEP_WIDTHS_BY_METHOD = {
    'local_split_entropy': (8, 16, 32, 64, 128),
    'fingerprint_distribution_entropy': (
        8, 16, 32, 64, 128, 256, 512, 1024,
    ),
}
SWEEP_NGRAM_SIZES = (2, 3, 4)
SWEEP_MAX_BLOCK_FREQUENCIES = (0.1, 0.25, 0.5, 1.0)
# The minimum cutoff is applied only by fingerprint_distribution_entropy.
SWEEP_MIN_BLOCK_FREQUENCIES_BY_METHOD = {
    'local_split_entropy': (0.0,),
    'fingerprint_distribution_entropy': (0.0, 0.01, 0.05),
}
# Use a one-element tuple to run either method alone, or keep both to compare them.
SWEEP_FEATURE_SELECTION_METHODS = (
    'local_split_entropy',
    'fingerprint_distribution_entropy',
)
# One label per sweep invocation keeps prior sweep directories intact while
# linking all parameter configurations produced by this invocation.
SWEEP_RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
SWEEP_DIR = PROJECT_ROOT / f'ceb_imdb_block_skipping_sweep_{SWEEP_RUN_ID}'
SWEEP_COLUMNS = [
    'feature_selection_method', 'ngram_size', 'min_block_frequency',
    'max_block_frequency',
    'fingerprint_width',
    'metadata_size_bytes',
    'mean_unnecessary_block_read_ratio', 'zero_bit_query_count',
    'query_count', 'total_candidate_partitions', 'total_pruned_partitions',
    'total_ground_truth_partitions', 'false_positive_partition_count',
    'false_negative_partition_count', 'run_directory',
]

assert all(
    widths and tuple(sorted(set(widths))) == widths
    and all(width > 0 for width in widths)
    for widths in SWEEP_WIDTHS_BY_METHOD.values()
)
assert SWEEP_NGRAM_SIZES and all(size > 0 for size in SWEEP_NGRAM_SIZES)
assert SWEEP_MAX_BLOCK_FREQUENCIES
assert all(0 < frequency <= 1 for frequency in SWEEP_MAX_BLOCK_FREQUENCIES)
assert all(
    frequencies and all(0 <= frequency <= 1 for frequency in frequencies)
    for frequencies in SWEEP_MIN_BLOCK_FREQUENCIES_BY_METHOD.values()
)
assert SWEEP_FEATURE_SELECTION_METHODS
assert set(SWEEP_FEATURE_SELECTION_METHODS) <= FINGERPRINT_FEATURE_SELECTION_METHODS
assert set(SWEEP_FEATURE_SELECTION_METHODS) <= set(SWEEP_WIDTHS_BY_METHOD)
assert set(SWEEP_FEATURE_SELECTION_METHODS) <= set(
    SWEEP_MIN_BLOCK_FREQUENCIES_BY_METHOD
)

_original_sweep_globals = (
    RUN_DIR, OUTPUT_DIR, FINGERPRINT_DIR, FINGERPRINT_WIDTHS,
    FINGERPRINT_NGRAM_SIZE, FINGERPRINT_FEATURE_SELECTION,
    FINGERPRINT_MIN_BLOCK_FREQUENCY, FINGERPRINT_MAX_BLOCK_FREQUENCY,
    NGRAM_GENERATOR,
)
SWEEP_ROWS = []
try:
    for (FINGERPRINT_FEATURE_SELECTION, FINGERPRINT_NGRAM_SIZE,
         FINGERPRINT_MIN_BLOCK_FREQUENCY,
         FINGERPRINT_MAX_BLOCK_FREQUENCY) in (
        (method, size, min_frequency, max_frequency)
        for method in SWEEP_FEATURE_SELECTION_METHODS
        for size in SWEEP_NGRAM_SIZES
        for min_frequency in SWEEP_MIN_BLOCK_FREQUENCIES_BY_METHOD[method]
        for max_frequency in SWEEP_MAX_BLOCK_FREQUENCIES
        if min_frequency <= max_frequency
    ):
        min_frequency_label = format(
            FINGERPRINT_MIN_BLOCK_FREQUENCY, 'g'
        ).replace('.', 'p')
        max_frequency_label = format(
            FINGERPRINT_MAX_BLOCK_FREQUENCY, 'g'
        ).replace('.', 'p')
        RUN_DIR = (SWEEP_DIR / FINGERPRINT_FEATURE_SELECTION
                   / f'ngram_{FINGERPRINT_NGRAM_SIZE}'
                   / f'min_block_frequency_{min_frequency_label}'
                   / f'max_block_frequency_{max_frequency_label}')
        OUTPUT_DIR = RUN_DIR / 'block_skipping_results'
        FINGERPRINT_DIR = RUN_DIR / 'block_infix_fingerprints'
        FINGERPRINT_WIDTHS = SWEEP_WIDTHS_BY_METHOD[FINGERPRINT_FEATURE_SELECTION]
        NGRAM_GENERATOR = NGramGenerator(
            FINGERPRINT_NGRAM_SIZE, ascii_only=FINGERPRINT_ASCII_ONLY
        )
        print(
            f'\n=== method {FINGERPRINT_FEATURE_SELECTION}; '
            f'n-gram size {FINGERPRINT_NGRAM_SIZE}; block frequency '
            f'[{FINGERPRINT_MIN_BLOCK_FREQUENCY:g}, '
            f'{FINGERPRINT_MAX_BLOCK_FREQUENCY:g}]; widths {FINGERPRINT_WIDTHS} ==='
        )
        (sweep_queries, sweep_skipped, sweep_truths, sweep_versions,
         sweep_partition_ids, sweep_matches, sweep_results) = evaluate_fingerprints()
        create_exports(sweep_results, sweep_versions, sweep_partition_ids)

        for version in sweep_versions:
            rows = [
                row for row in sweep_results
                if row['metadata_version'] == version.metadata_version
            ]
            ratios = [
                row['false_positive_partition_count']
                / (row['partition_file_count'] - row['ground_truth_partition_count'])
                for row in rows
                if row['partition_file_count'] > row['ground_truth_partition_count']
            ]
            SWEEP_ROWS.append({
                'feature_selection_method': FINGERPRINT_FEATURE_SELECTION,
                'ngram_size': FINGERPRINT_NGRAM_SIZE,
                'min_block_frequency': FINGERPRINT_MIN_BLOCK_FREQUENCY,
                'max_block_frequency': FINGERPRINT_MAX_BLOCK_FREQUENCY,
                'fingerprint_width': rows[0]['query_fingerprint_width'],
                'metadata_size_bytes': version.metadata_size_bytes,
                'mean_unnecessary_block_read_ratio': mean(ratios),
                'zero_bit_query_count': sum(
                    row['query_fingerprint_ones'] == 0 for row in rows
                ),
                'query_count': len(rows),
                'total_candidate_partitions': sum(
                    row['metadata_partition_count'] for row in rows
                ),
                'total_pruned_partitions': sum(
                    row['pruned_partition_count'] for row in rows
                ),
                'total_ground_truth_partitions': sum(
                    row['ground_truth_partition_count'] for row in rows
                ),
                'false_positive_partition_count': sum(
                    row['false_positive_partition_count'] for row in rows
                ),
                'false_negative_partition_count': sum(
                    row['false_negative_partition_count'] for row in rows
                ),
                'run_directory': str(RUN_DIR.resolve()),
            })
finally:
    (RUN_DIR, OUTPUT_DIR, FINGERPRINT_DIR, FINGERPRINT_WIDTHS,
     FINGERPRINT_NGRAM_SIZE, FINGERPRINT_FEATURE_SELECTION,
     FINGERPRINT_MIN_BLOCK_FREQUENCY, FINGERPRINT_MAX_BLOCK_FREQUENCY,
     NGRAM_GENERATOR) = _original_sweep_globals

SWEEP_SUMMARY_PATH = SWEEP_DIR / 'fingerprint_sweep_summary.csv'
write_csv(SWEEP_SUMMARY_PATH, SWEEP_COLUMNS, SWEEP_ROWS)
print(f'Wrote {len(SWEEP_ROWS)} sweep points to {SWEEP_SUMMARY_PATH}')
SWEEP_SUMMARY_PATH



=== method local_split_entropy; n-gram size 2; block frequency [0, 0.1]; widths (8, 16, 32, 64, 128) ===
Building concatenated block strings [1/1]: title.title
  155 blocks; selected 128/128 features with local_split_entropy
  feature order: [' *', 'q/', 's4', "')", 'k9', '0?', '*e', '1%', '#@', '6b', 'n@', '4m', 'm6', '&c', '8r', 'b0', '!t', 'k4', 'bx', '&t', 's(', 'm&', 'h3', '(q', '0p', '!i', '$a', '*n', ")'", '_e', ' <', '0b', '0h', '@k', 'n"', '"n', '*2', '2?', 'c6', 'g1', '"a', '"p', '#!', ' %', ' _', '!d', '"m', '$9', "('", '8h', '#p', '$8', '&2', '&e', '&r', '&w', '(.', '1o', '1x', '5b', '5d', '?,', 'b1', 'g2', '  ', ' !', ' )', ' /', ' ;', ' `', ' {', '!"', '!*', '!+', '!,', '!l', '!n', '!o', '".', '"2', '"b', '"d', '"f', '"h', '"i', '"k', '#*', '#b', '#k', ' >', ' ]', '!#', '!%', '!-', '!1', '!2', '!3', '!a', '!b', '!e', '!g', '!j', '!m', '!p', '!r', '!s', '!v', '!w', '"/', '":', '"?', '"c', '"e', '"j', '"l', '"o', '"r', '"w', '"x', '"y', '#$', '#%', '#&', '#.', '#?', '#c', 

KeyboardInterrupt: 